# YOLOv11 — Shelf Void Detection (Free GPU Training)

**Roman Urdu guide (beginner ke liye):**

Yeh notebook aapke Roboflow dataset zip se **free me** YOLOv11 model train karta hai — Google Colab ke free T4 GPU pe. Zero paise, zero Roboflow credits.

## Kaise use karein (5 steps):
1. **GPU on karo** → Runtime menu → *Change runtime type* → **T4 GPU** → Save
2. **dataset zip upload karo** → left sidebar 📁 → `/content` → apni zip upload karo (koi bhi naam chalega, neeche auto-detect ho jayega)
3. **Run All** karo (Ctrl+F9 ya Runtime → Run all)
4. **Wait** ~25-35 min (yolo11s, 120 epochs)
5. **best.pt download** → niche wali cells automatically download karenge

## Classes — auto-detected from data.yaml
Notebook classes ko **data.yaml se automatically padhta hai**, taaki koi bhi dataset version (2-class, 3-class, future v7/v8...) ke saath kaam kare. Current **Supermarket Shelf Void Space** dataset:
- `0 = missing` (khali shelf / void space — **unoccupied**)
- `1 = product` (bhara shelf — **occupied**)

> ⚠️ Class IDs positional hain — yeh order aapke Roboflow export ke `data.yaml` se aata hai, isliye notebook me **hardcode nahi kiya**. Naam badal jayein (jaise `shelf-void` vs `missing`) tab bhi notebook sahi chalega.

---


## Step 1 — GPU check karo

Cell run karne pe **Tesla T4** ya koi GPU dikhni chahiye. Agar `command not found` ya CPU aaye toh: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - GPU on karo!")


## Step 2 — Ultralytics (YOLOv11) install karo

Official YOLOv11 package. Install me ~1 min lagta hai.

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import os, shutil, yaml, glob
print("Ultralytics installed")


## Step 3 — Dataset upload + setup

### Pehle zip upload karo:
Left sidebar me **folder icon 📁** → `/content` → **Upload** → apna file select karo. **Naam `dataset.zip` rakhna** (ya neeche `ZIP_NAME` badal do).

Upload ~5 min lagta hai. Upload complete hone ke baad hi yeh cell run karo.

**Download kahan se:** Roboflow → aapka project → Version → *Download Dataset* → **YOLOv8** ya **YOLOv11** format (dono same label format use karte hain — ultralytics dono padh leta hai).

In [ ]:
ZIP_NAME = "dataset.zip"   # default naam. Alag hai toh ya toh yahan likho, ya neeche auto-detect chalne do.
ROOT = "/content"
DATA_DIR = ROOT + "/shelf_dataset"

# Clean previous
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(ROOT, ZIP_NAME)
# Agar exact naam na mile, toh /content me koi bhi .zip auto-detect kar lo
# (e.g. "Supermarket Shelf Void Space.yolov11.zip" — spaces wala naam bhi chalta hai)
if not os.path.exists(zip_path):
    zips = sorted(f for f in os.listdir(ROOT) if f.lower().endswith(".zip"))
    assert zips, "/content me koi .zip nahi mila! Pehle zip upload karo (sidebar 📁 se Upload)."
    zip_path = os.path.join(ROOT, zips[0])
    if len(zips) > 1:
        print("⚠️  Multiple zips milin, pehla use kar rahe hain:", zips[0], "(ZIP_NAME badal do agar koi aur chahiye)")
    else:
        print("Auto-detected zip:", zips[0])

# Unzip
!unzip -q "{zip_path}" -d "{DATA_DIR}"
print("Unzipped to:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))


In [ ]:
# Structure check + data.yaml path fix
# Roboflow export me paths "../train/images" ki hote hain (relative-wrong) -> hum absolute likhte hain.
# LEKIN nc + names ko original export se padhte hain (hardcode NAHI) -> labels ke saath hamesha match.
yaml_files = glob.glob(DATA_DIR + "/**/data.yaml", recursive=True)
assert yaml_files, "data.yaml nahi mila"
data_yaml_path = os.path.abspath(yaml_files[0])
base_dir = os.path.dirname(data_yaml_path)

# Roboflow classes read karo = single source of truth (future versions ke liye safe)
with open(data_yaml_path) as f:
    src = yaml.safe_load(f) or {}
names = src.get("names")
if isinstance(names, dict):                       # kuch exports {0: 'x', ...} dete hain
    names = [names[i] for i in sorted(names)]
if not names:                                     # data.yaml me names nahi -> export toota hua
    raise RuntimeError("data.yaml me 'names' field missing! Roboflow export dobara check karo.")
nc = src.get("nc") or len(names)
assert nc == len(names), "nc=%s but names len=%d — data.yaml check karo" % (nc, len(names))

cfg = {
    "train": base_dir + "/train/images",
    "val":   base_dir + "/valid/images",
    "test":  base_dir + "/test/images",
    "nc": nc,
    "names": names,
}
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("data.yaml fixed:", data_yaml_path)
print(open(data_yaml_path).read())

for split in ["train", "valid", "test"]:
    imgs = glob.glob(base_dir + "/" + split + "/images/*.jpg")
    labs = glob.glob(base_dir + "/" + split + "/labels/*.txt")
    print("%-6s: %d images, %d labels" % (split, len(imgs), len(labs)))

# Sanity: koi label class id nc ke bahar toh nahi? (agar haan toh training galat/classes drop honge)
max_id = -1
for lab in glob.glob(base_dir + "/**/labels/*.txt", recursive=True):
    with open(lab) as f:
        for line in f:
            t = line.split()
            if t and t[0].lstrip("-").isdigit():
                max_id = max(max_id, int(t[0]))
print("Max class id in labels:", max_id, "| nc:", nc)
assert max_id < nc, "Label me class id %d hai but nc=%d — data.yaml me names check karo!" % (max_id, nc)
print("Class map:", {i: names[i] for i in range(nc)})


## Step 3.5 — Class Imbalance: smart oversampling (auto) ⭐

Dataset me `product` boxes bohat zyada hain, `missing` (void) boxes kam. Lekin oversampling **har dataset pe kaam nahi karta** — isliye notebook pehle check karta hai ke fayda hoga ya nahi, phir decide karta hai.

**Rule:** Oversampling sirf tab fayda deta hai jab koi class **kuch (sab nahi) images** me ho. Agar ek class **har image me** ho (≈100% coverage), toh uske images duplicate karne se poora dataset uniform ×N badh jata hai — **rebalance kuch nahi hota, bas training time waste**.

**Is dataset ke liye:** `missing` aur `product` dono ~100% train images me hain, isliye image-level oversampling **auto-skip** ho jayega. Yeh sahi behaviour hai, galat nahi.

> 💡 **Asli imbalance box-level hai** (`missing` ≈ 5% boxes). Image duplication se ye fix nahi hota. Asli fixes:
> - Inference par **CONFIDENCE thoda kam** (0.20–0.25) → zyada void recall
> - **Aur missing/void images** collect karo (sabse effective)
>
> Notebook **class naam/id se kuch hardcode nahi karta** — agar koi class future dataset me kam images me hui, toh uska oversample automatically ho jayega.


In [ ]:
# === OVERSAMPLING: sirf tab karo jab koi class "image-level minority" ho ===
# Agar koi class HAR image me ho, uska oversample rebalance nahi karta (bas time waste).
# Isliye pehle har class ka image-coverage nikaalte hain, phir sirf un classes ko
# oversample karte hain jo kam images me mile (MIN_COVERAGE se kam). Class naam/id
# se kuch hardcode NAHI — kisi bhi dataset version pe sahi chalta hai.
OVERSAMPLE_FACTOR = 3     # image itni baar rakho (3 = original + 2 copies)
MIN_COVERAGE    = 0.85    # class jo >85% train images me hai -> oversample fayda nahi

train_lab_dir = base_dir + "/train/labels"
train_labs = glob.glob(train_lab_dir + "/*.txt")
n_train = len(train_labs)

# Har class kitne images me present hai (per-image; ek box kaafi hai)
class_img_count = {i: 0 for i in range(nc)}
for lab in train_labs:
    present = set()
    with open(lab) as f:
        for line in f:
            t = line.split()
            if t and t[0].lstrip("-").isdigit():
                present.add(int(t[0]))
    for cid in present:
        class_img_count[cid] += 1

print("Per-class image coverage (train, %d images):" % n_train)
for cid in range(nc):
    cov = class_img_count[cid] / max(1, n_train)
    print("  id %d %-12s : %d images (%.0f%%)" % (cid, names[cid], class_img_count[cid], 100 * cov))

# Minority = wo class jiski coverage MIN_COVERAGE se kam ho
target_ids = {cid for cid in range(nc)
              if class_img_count[cid] / max(1, n_train) < MIN_COVERAGE}

if not target_ids:
    print("\nℹ️  Koi image-level minority nahi mila (sab classes ~har image me hain).")
    print("    Oversampling SKIP — is dataset pe fayda nahi deta. Imbalance box-level hai;")
    print("    uska fix = inference CONFIDENCE kam karna + aur void images collect karna.")
else:
    print("\nOversample class ids:", sorted(target_ids),
          "(%s) x%d" % ([names[i] for i in sorted(target_ids)], OVERSAMPLE_FACTOR))
    train_img_dir = base_dir + "/train/images"

    def has_target(lab_path):
        with open(lab_path) as f:
            for line in f:
                t = line.split()
                if t and t[0].isdigit() and int(t[0]) in target_ids:
                    return True
        return False

    cand = []
    for img in glob.glob(train_img_dir + "/*.jpg"):
        lab = os.path.join(train_lab_dir, os.path.splitext(os.path.basename(img))[0] + ".txt")
        if os.path.exists(lab) and has_target(lab):
            cand.append((img, lab))

    before_n = len(glob.glob(train_img_dir + "/*.jpg"))
    added = 0
    for img, lab in cand:
        for k in range(OVERSAMPLE_FACTOR - 1):   # original pehle se hai, sirf copies add karo
            stem, ext = os.path.splitext(img)
            img_dup = stem + "_dup" + str(k) + ext
            lab_dup = os.path.splitext(lab)[0] + "_dup" + str(k) + ".txt"
            if not os.path.exists(img_dup):
                shutil.copy(img, img_dup)
                shutil.copy(lab, lab_dup)
                added += 1
    after_n = len(glob.glob(train_img_dir + "/*.jpg"))
    print("Oversample-candidate train images:", len(cand))
    print("Oversample x%d: %d copies add kiye" % (OVERSAMPLE_FACTOR, added))
    print("Train images: %d -> %d" % (before_n, after_n))


## Step 4 — Training (improved settings)

Void (`missing`) recall behtar karne ke liye ye settings:

| Setting | Value | Kyun |
|---|---|---|
| Model | `yolo11s.pt` (small) | Nano se zyada capacity → voids behtar seekhega |
| Epochs | `120`, patience `35` | Acha converge, jaldi early-stop na ho |
| **imgsz** | **`960`** (640 se upar) | Patli/thin void slivers 640 pe kho jaate hain |
| **close_mosaic** | **`15`** | Last 15 epochs me mosaic band → void boxes ki localization behtar |
| **cls loss** | **`0.8`** (default 0.5) | Minority `missing` class (~5% boxes) drown/ignore na ho |
| Oversampling | ✅ Step 3.5 (**auto**) | Sirf tab jab koi class image-level minority ho (is dataset pe auto-skip) |

> Training time ~30-45 min (yolo11s, imgsz 960). Agar **`OutOfMemory (OOM)`** aaye toh niche `BATCH = 8` ko `4` kar do.
> Fast chahiye toh `MODEL_VARIANT = "yolo11n.pt"` (par void recall thoda kam).


In [ ]:
# === CONFIG ===
MODEL_VARIANT = "yolo11s.pt"   # yolo11n.pt (fast) / yolo11s.pt (better, default) / yolo11m.pt (best, slow)
EPOCHS = 120
IMG_SIZE = 960                  # 640 -> 960: patli/thin void slivers 640 pe kho jaate hain (small-object recall)
BATCH = 8                       # imgsz double -> batch aadha (OOM safety). OOM aaye toh 4 karo.
PATIENCE = 35
RUN_NAME = "supermarket_void_2cls_v960"

print("Training: %s | %d epochs | batch %d | imgsz %d | %d classes" % (
    MODEL_VARIANT, EPOCHS, BATCH, IMG_SIZE, nc))

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    patience=PATIENCE,
    cache=True,
    plots=True,
    exist_ok=True,
    # --- Void-recall helpers (missing class minority ~5% boxes hai) ---
    close_mosaic=15,   # last 15 epochs me mosaic band -> bade void boxes ki localization behtar hoti hai
    cls=0.8,           # cls loss upar (default 0.5) -> minority void class ignore/drown na ho
    mosaic=1.0,        # default; chote dataset me variety ke liye
    fliplr=0.5,        # horizontal flip (default) -> ~2x data effect
    # copy_paste=0.1,  # (OPTIONAL) void boxes duplicate karke imbalance kam karta hai; masks chahiye warna artifacts
)
print("✅ TRAINING DONE")


## Step 5 — Results + Metrics dekho

Sabse important: **`missing`** (void) class ka recall — model khali shelf kitni baar pakadta hai. Per-class numbers trained model ke `model.names` se aate hain — hardcode nahi.

> Ye eval **test split** par hoti hai (sabse honest numbers — train se alag images).


In [ ]:
# Validation on test set
metrics = model.val(data=data_yaml_path, split="test", plots=True)
print("\n=== OVERALL ===")
print("mAP50      : %.4f" % metrics.box.map50)
print("mAP50-95   : %.4f" % metrics.box.map)
print("Precision  : %.4f" % metrics.box.mp)
print("Recall     : %.4f" % metrics.box.mr)

print("\n=== PER-CLASS (trained model.names se) ===")
cls_names = [model.names[i] for i in range(len(model.names))]
for i, n in enumerate(cls_names):
    print("%-12s: mAP50=%.4f  P=%.4f  R=%.4f" % (n, metrics.box.ap50[i], metrics.box.p[i], metrics.box.r[i]))

## Step 6 — Best Model Download

In [ ]:
best_model_path = "runs/detect/" + RUN_NAME + "/weights/best.pt"
assert os.path.exists(best_model_path), "best.pt nahi mila — training check karo"

from google.colab import files
files.download(best_model_path)
print("best.pt download ho raha hai... (browser popup)")
print("Size: %.1f MB" % (os.path.getsize(best_model_path)/1024/1024))


## Step 7 — Training Graphs

In [ ]:
from IPython.display import Image, display
run_dir = "runs/detect/" + RUN_NAME
for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = run_dir + "/" + img_name
    if os.path.exists(p):
        print("\n--- " + img_name + " ---")
        display(Image(filename=p, width=600))


---
*Notebook • YOLOv11 • Free Colab GPU • classes auto-read from data.yaml • smart class-imbalance oversampling*
